In [1]:

# Day 4 - Task 3
# Pipeline + PCA + Data Leakage Demonstration

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# ==========================================
# APPROACH 1: CORRECT METHOD - NO DATA LEAKAGE
# ==========================================

# 2. Split data first
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Build Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("classifier", LogisticRegression(max_iter=5000))
])

# 4. Train the pipeline
pipeline.fit(X_train, y_train)

# 5. Predict on test data
pred_correct = pipeline.predict(X_test)

# 6. Calculate correct score
score_correct = accuracy_score(y_test, pred_correct)

print("===== Correct Pipeline =====")
print("Accuracy:", round(score_correct, 4))
print("PCA Components:",
      pipeline.named_steps["pca"].n_components_)


# ==========================================
# APPROACH 2: DATA LEAKAGE
# Scaling and PCA BEFORE splitting
# ==========================================

# 7. Fit scaler on the entire dataset (INCORRECT)
scaler_leak = StandardScaler()
X_scaled_leak = scaler_leak.fit_transform(X)

# 8. Fit PCA on the entire dataset (INCORRECT)
pca_leak = PCA(n_components=0.95)
X_pca_leak = pca_leak.fit_transform(X_scaled_leak)

# 9. Split after preprocessing
X_train_leak, X_test_leak, y_train_leak, y_test_leak = (
    train_test_split(
        X_pca_leak, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

# 10. Train classifier
model_leak = LogisticRegression(max_iter=5000)
model_leak.fit(X_train_leak, y_train_leak)

# 11. Predict
pred_leak = model_leak.predict(X_test_leak)

# 12. Calculate leakage score
score_leak = accuracy_score(y_test_leak, pred_leak)

print("\n===== Pipeline with Data Leakage =====")
print("Accuracy:", round(score_leak, 4))
print("PCA Components:", pca_leak.n_components_)

# ==========================================
# 13. COMPARE SCORES
# ==========================================

print("\n===== Score Comparison =====")
print("Correct Pipeline Score:", round(score_correct, 4))
print("Leakage Approach Score:", round(score_leak, 4))
print("Score Difference:",
      round(score_leak - score_correct, 4))


===== Correct Pipeline =====
Accuracy: 0.9737
PCA Components: 10

===== Pipeline with Data Leakage =====
Accuracy: 0.9649
PCA Components: 10

===== Score Comparison =====
Correct Pipeline Score: 0.9737
Leakage Approach Score: 0.9649
Score Difference: -0.0088
